# 🏐 Lab 1 — Web Crawling + NER
**Course:** Web Mining & Semantics  
**Domain:** International Men's Volleyball (FIVB)  
**Sources:** Wikipedia API + Wikidata SPARQL  

### Pipeline Overview
```
Wikipedia API ──► Text Cleaning ──► NER (spaCy) ──► extracted_knowledge.csv
Wikidata SPARQL ──────────────────────────────────► wikidata_output.jsonl
NER output ──────────────────────────────────────► Relation Extraction
```

### ⚖️ Web Ethics & Crawling Policy
This crawler follows responsible web scraping practices:
- **Wikipedia API** is used instead of direct HTML scraping — this is the officially recommended access method per Wikipedia's [Terms of Use](https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use)
- **Wikidata SPARQL endpoint** is public and free to query with a proper `User-Agent` header
- **Rate limiting**: 0.5s delay between Wikipedia API calls to avoid overloading servers
- **User-Agent**: identifies the bot as an academic project as required by Wikipedia's robots.txt
- **robots.txt compliance**: Wikipedia's robots.txt allows API access; Wikidata explicitly permits SPARQL queries

### Outputs
| File | Description |
|------|-------------|
| `crawler_output.jsonl` | Cleaned Wikipedia text + metadata per page |
| `wikidata_output.jsonl` | Structured Wikidata query results |
| `extracted_knowledge.csv` | Named entities (type, text, sentence context, source) |
| `extracted_relations.csv` | Candidate triples (subject, verb, object) |

## ⚙️ 0. Setup & Installations

In [ ]:
# Install dependencies
!pip install -q trafilatura httpx SPARQLWrapper pandas wikipedia-api
!pip install -q spacy
!python -m spacy download en_core_web_trf -q

In [ ]:
# Mount Google Drive to save outputs
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/volleyball-kg/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Imports
import json
import time
import httpx
import trafilatura
import pandas as pd
import spacy
import wikipediaapi
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.parse import urlparse

# Load spaCy model
nlp = spacy.load('en_core_web_trf')
print('All imports OK ✅')

## 🌐 1. Phase 1 — Wikipedia Crawling

In [ ]:
# ── Seed URLs ──────────────────────────────────────────────────────────────
# Covers: national teams, players, tournaments, clubs
SEED_URLS = [
    # National Teams
    'https://en.wikipedia.org/wiki/Brazil_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/France_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Poland_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Italy_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/United_States_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Japan_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Argentina_men%27s_national_volleyball_team',
    # Tournaments
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_Nations_League',
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_World_Championship',
    'https://en.wikipedia.org/wiki/Volleyball_at_the_Summer_Olympics',
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_World_Cup',
    # Players
    'https://en.wikipedia.org/wiki/Earvin_Ngapeth',
    'https://en.wikipedia.org/wiki/Wilfredo_Le%C3%B3n',
    'https://en.wikipedia.org/wiki/Nimir_Abdel-Aziz',
    'https://en.wikipedia.org/wiki/Giba',
    'https://en.wikipedia.org/wiki/Bartosz_Kurek',
    'https://en.wikipedia.org/wiki/Ivan_Zaytsev',
    'https://en.wikipedia.org/wiki/Matthew_Anderson_(volleyball)',
    'https://en.wikipedia.org/wiki/Bruno_Rezende',
    # Clubs
    'https://en.wikipedia.org/wiki/Trentino_Volley',
    'https://en.wikipedia.org/wiki/Sir_Sicoma_Monini_Perugia',
    'https://en.wikipedia.org/wiki/Paris_Volley',
    # Federations
    'https://en.wikipedia.org/wiki/FIVB',
    'https://en.wikipedia.org/wiki/Confédération_Européenne_de_Volleyball',
]

print(f'Total seed URLs: {len(SEED_URLS)}')

In [ ]:
# Use Wikipedia API — official access method, no 403 issues
# User-Agent identifies us as an academic project (required by Wikipedia ToS)
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='VolleyballKG-Bot/1.0 (academic research project, non-commercial)'
)

WIKI_PAGES = [
    # === National Teams ===
    "Brazil men's national volleyball team",
    "France men's national volleyball team",
    "Poland men's national volleyball team",
    "Italy men's national volleyball team",
    "United States men's national volleyball team",
    "Japan men's national volleyball team",
    "Russia men's national volleyball team",
    "Argentina men's national volleyball team",
    "Serbia men's national volleyball team",
    "Cuba men's national volleyball team",
    "Slovenia men's national volleyball team",
    # === Tournaments (specific editions to get richer content) ===
    "2024 FIVB Volleyball Nations League – Men",
    "2023 FIVB Volleyball Nations League – Men",
    "2022 FIVB Volleyball World Championship – Men",
    "2018 FIVB Volleyball World Championship",
    "Volleyball at the 2024 Summer Olympics – Men's tournament",
    "Volleyball at the 2020 Summer Olympics – Men's tournament",
    "Volleyball at the Summer Olympics",
    "2021 FIVB Volleyball World Cup",
    "CEV Champions League Volleyball – Men 2023–24",
    # === Players ===
    "Earvin Ngapeth",
    "Wilfredo León",
    "Nimir Abdel-Aziz",
    "Giba",
    "Bartosz Kurek",
    "Ivan Zaytsev",
    "Matthew Anderson (volleyball)",
    "Bruno Rezende",
    "Micah Christenson",
    "Maxim Mikhaylov",
    "Aleksandar Atanasijević",
    "Facundo Conte",
    "Yoandy Leal",
    "Osmany Juantorena",
    "Robertlandy Simón",
    # === Clubs ===
    "Trentino Volley",
    "Cucine Lube Civitanova",
    "Zenit Kazan",
    "Asseco Resovia",
    "Paris Volley",
    # === Federations & Orgs ===
    "FIVB",
    "Confédération Européenne de Volleyball",
    "Volleyball",
]

def is_useful_page(text: str, min_words: int = 200) -> bool:
    """Check if extracted text meets minimum length threshold.
    Lowered to 200 words to capture shorter but valid pages (player bios, small clubs).
    """
    return len(text.split()) >= min_words

def fetch_wikipedia_page(title: str) -> dict | None:
    """
    Fetch a Wikipedia page by title using the official Wikipedia API.
    Respects rate limiting and ToS — no direct HTML scraping.
    Returns dict with title, url, text, word_count — or None if not useful.
    """
    try:
        page = wiki.page(title)
        if not page.exists():
            print(f'  [SKIP] Page not found: {title}')
            return None
        text = page.text
        if not is_useful_page(text):
            print(f'  [SKIP] Too short ({len(text.split())} words): {title}')
            return None
        return {
            'url': page.fullurl,
            'title': title,
            'text': text,
            'word_count': len(text.split())
        }
    except Exception as e:
        print(f'  [ERROR] {title}: {e}')
        return None

print(f'Total pages to fetch: {len(WIKI_PAGES)}')
print(f'Minimum word threshold: 200 words')

In [ ]:
# Crawl all Wikipedia pages via API
crawled_pages = []
DELAY_SECONDS = 0.5

print('Starting Wikipedia crawl via API...\n')
for i, title in enumerate(WIKI_PAGES):
    print(f'[{i+1}/{len(WIKI_PAGES)}] Fetching: {title}')
    result = fetch_wikipedia_page(title)
    if result:
        crawled_pages.append(result)
        print(f'  [OK] {result["word_count"]} words')
    time.sleep(DELAY_SECONDS)

print(f'\n✅ Successfully crawled: {len(crawled_pages)}/{len(WIKI_PAGES)} pages')

In [ ]:
# ── Save to JSONL ──────────────────────────────────────────────────────────
JSONL_PATH = f'{OUTPUT_DIR}/crawler_output.jsonl'

with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for page in crawled_pages:
        f.write(json.dumps(page, ensure_ascii=False) + '\n')

print(f'Saved {len(crawled_pages)} pages → {JSONL_PATH}')

## 🗺️ 2. Phase 1b — Wikidata SPARQL Fetcher

In [ ]:
def query_wikidata(sparql_query: str) -> list[dict]:
    """Execute a SPARQL query on Wikidata and return results as a list of dicts."""
    endpoint = SPARQLWrapper('https://query.wikidata.org/sparql')
    endpoint.setQuery(sparql_query)
    endpoint.setReturnFormat(JSON)
    endpoint.addCustomHttpHeader('User-Agent', 'VolleyballKG-Bot/1.0 (academic project)')
    results = endpoint.query().convert()
    return results['results']['bindings']

In [ ]:
# ── Query 1: Male volleyball players ──────────────────────────────────────
QUERY_PLAYERS = """
SELECT DISTINCT ?player ?playerLabel ?nationalityLabel ?birthDate ?positionLabel WHERE {
  ?player wdt:P31 wd:Q5 ;                      # is a human
          wdt:P106 wd:Q15117302 ;               # occupation: volleyball player
          wdt:P21 wd:Q6581097 .                 # gender: male
  OPTIONAL { ?player wdt:P27 ?nationality . }
  OPTIONAL { ?player wdt:P569 ?birthDate . }
  OPTIONAL { ?player wdt:P413 ?position . }     # playing position
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 500
"""

print('Querying Wikidata: volleyball players...')
players_raw = query_wikidata(QUERY_PLAYERS)
print(f'Found {len(players_raw)} players')

In [ ]:
# ── Query 2: Men's national volleyball teams ───────────────────────────────
QUERY_TEAMS = """
SELECT DISTINCT ?team ?teamLabel ?countryLabel ?federationLabel WHERE {
  ?team wdt:P31 wd:Q6979593 .                   # men's national volleyball team
  OPTIONAL { ?team wdt:P17 ?country . }
  OPTIONAL { ?team wdt:P118 ?federation . }      # league/federation
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 200
"""

print('Querying Wikidata: national teams...')
teams_raw = query_wikidata(QUERY_TEAMS)
print(f'Found {len(teams_raw)} teams')

In [ ]:
# ── Query 3: Volleyball tournaments/competitions ───────────────────────────
QUERY_TOURNAMENTS = """
SELECT DISTINCT ?tournament ?tournamentLabel ?organizerLabel ?inceptionYear WHERE {
  ?tournament wdt:P31/wdt:P279* wd:Q13406554 .  # volleyball competition
  OPTIONAL { ?tournament wdt:P664 ?organizer . }
  OPTIONAL { ?tournament wdt:P571 ?inception .
             BIND(YEAR(?inception) AS ?inceptionYear) }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 100
"""

print('Querying Wikidata: tournaments...')
tournaments_raw = query_wikidata(QUERY_TOURNAMENTS)
print(f'Found {len(tournaments_raw)} tournaments')

In [ ]:
# Query 4: Player <-> National Team memberships (via country for sport)
QUERY_MEMBERSHIPS = """
SELECT DISTINCT ?playerLabel ?teamLabel ?countryLabel WHERE {
  ?player wdt:P31 wd:Q5 ;
          wdt:P106 wd:Q15117302 ;
          wdt:P21 wd:Q6581097 ;
          wdt:P1532 ?country .
  ?team wdt:P31 wd:Q6979593 ;
        wdt:P17 ?country .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 500
"""

print('Querying Wikidata: player-team memberships...')
try:
    memberships_raw = query_wikidata(QUERY_MEMBERSHIPS)
    print(f'Found {len(memberships_raw)} memberships')
except Exception as e:
    print(f'Query failed: {e} — using empty list')
    memberships_raw = []


In [ ]:
# ── Save Wikidata results as structured JSONL ──────────────────────────────
WIKIDATA_PATH = f'{OUTPUT_DIR}/wikidata_output.jsonl'

wikidata_results = {
    'players': players_raw,
    'teams': teams_raw,
    'tournaments': tournaments_raw,
    'memberships': memberships_raw
}

with open(WIKIDATA_PATH, 'w', encoding='utf-8') as f:
    for category, records in wikidata_results.items():
        for record in records:
            row = {'source': 'wikidata', 'category': category}
            row.update({k: v['value'] for k, v in record.items()})
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

total = sum(len(v) for v in wikidata_results.values())
print(f'Saved {total} Wikidata records → {WIKIDATA_PATH}')

## 🏷️ 3. Phase 2 — Named Entity Recognition (NER)

In [ ]:
# Entity types we care about for volleyball KG
TARGET_LABELS = {'PERSON', 'ORG', 'GPE', 'EVENT', 'DATE', 'LOC'}

def extract_entities(page: dict) -> list[dict]:
    """
    Run spaCy NER on a crawled page.
    Returns a list of entity records with type, text, sentence context, and source URL.
    """
    entities = []
    # Process in chunks to avoid memory issues with long texts
    text = page['text'][:50000]  # Cap at 50k chars for Colab memory
    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ not in TARGET_LABELS:
            continue
        # Filter out very short or very long entities (likely noise)
        if len(ent.text.strip()) < 2 or len(ent.text.split()) > 6:
            continue
        entities.append({
            'entity_text': ent.text.strip(),
            'entity_type': ent.label_,
            'source_url': page['url'],
            'source_title': page['title'],
            'sentence': ent.sent.text.strip()[:200]  # context snippet
        })

    return entities

In [ ]:
# ── Run NER on all crawled pages ───────────────────────────────────────────
all_entities = []

print('Running NER on crawled pages...\n')
for i, page in enumerate(crawled_pages):
    print(f'[{i+1}/{len(crawled_pages)}] Processing: {page["title"]}')
    entities = extract_entities(page)
    all_entities.extend(entities)
    print(f'  → {len(entities)} entities found')

print(f'\n✅ Total entities extracted: {len(all_entities)}')

## 🔗 4. Phase 2b — Relation Extraction (Dependency Parsing)

In [ ]:
def extract_relations(page: dict) -> list[dict]:
    """
    Extract candidate (subject, relation, object) triples using dependency parsing.
    Captures 4 patterns:
      1. nsubj → VERB → dobj       (e.g. 'France won the championship')
      2. nsubj → VERB → prep+pobj  (e.g. 'Ngapeth played for Modena')
      3. nsubj → VERB → attr       (e.g. 'Giba is a Brazilian player')
      4. Entity co-occurrence in same sentence (weaker signal, labeled 'co-occurs')
    """
    relations = []
    text = page['text'][:50000]
    doc = nlp(text)

    for sent in doc.sents:
        ents_in_sent = [e for e in sent.ents if e.label_ in TARGET_LABELS]
        if len(ents_in_sent) < 2:
            continue

        sent_relations = []

        for token in sent:
            if token.pos_ not in ('VERB', 'AUX'):
                continue

            subject_ent = None
            object_ent  = None

            for child in token.children:
                # Find subject entity
                if child.dep_ in ('nsubj', 'nsubjpass'):
                    for ent in ents_in_sent:
                        if ent.start <= child.i < ent.end:
                            subject_ent = ent

                # Find object entity — direct object
                if child.dep_ in ('dobj', 'attr', 'oprd'):
                    for ent in ents_in_sent:
                        if ent.start <= child.i < ent.end:
                            object_ent = ent

                # Find object entity — prepositional (played FOR France)
                if child.dep_ == 'prep':
                    for grandchild in child.children:
                        if grandchild.dep_ == 'pobj':
                            for ent in ents_in_sent:
                                if ent.start <= grandchild.i < ent.end:
                                    # Include preposition in relation label
                                    rel_label = f"{token.lemma_}_{child.text}"
                                    if subject_ent and subject_ent.text != ent.text:
                                        sent_relations.append({
                                            'subject': subject_ent.text,
                                            'subject_type': subject_ent.label_,
                                            'relation': rel_label,
                                            'object': ent.text,
                                            'object_type': ent.label_,
                                            'pattern': 'verb+prep',
                                            'sentence': sent.text.strip()[:200],
                                            'source_url': page['url']
                                        })

            if subject_ent and object_ent and subject_ent.text != object_ent.text:
                sent_relations.append({
                    'subject': subject_ent.text,
                    'subject_type': subject_ent.label_,
                    'relation': token.lemma_,
                    'object': object_ent.text,
                    'object_type': object_ent.label_,
                    'pattern': 'subj-verb-obj',
                    'sentence': sent.text.strip()[:200],
                    'source_url': page['url']
                })

        # Pattern 4: co-occurrence (fallback — pairs of entities in same sentence)
        if len(ents_in_sent) >= 2 and not sent_relations:
            for i in range(len(ents_in_sent)):
                for j in range(i+1, len(ents_in_sent)):
                    e1, e2 = ents_in_sent[i], ents_in_sent[j]
                    if e1.label_ != e2.label_:  # Only cross-type co-occurrences
                        sent_relations.append({
                            'subject': e1.text,
                            'subject_type': e1.label_,
                            'relation': 'co-occurs_with',
                            'object': e2.text,
                            'object_type': e2.label_,
                            'pattern': 'co-occurrence',
                            'sentence': sent.text.strip()[:200],
                            'source_url': page['url']
                        })

        relations.extend(sent_relations)

    return relations


In [ ]:
# ── Run relation extraction ────────────────────────────────────────────────
all_relations = []

print('Running relation extraction...\n')
for page in crawled_pages:
    rels = extract_relations(page)
    all_relations.extend(rels)

print(f'✅ Total candidate relations extracted: {len(all_relations)}')
# Preview
pd.DataFrame(all_relations).head(10)

## 💾 5. Export Final Deliverables

In [ ]:
# Export extracted_knowledge.csv
df_entities = pd.DataFrame(all_entities)

if df_entities.empty:
    print('WARNING: No entities extracted — check that crawled_pages is not empty')
    print(f'crawled_pages count: {len(crawled_pages)}')
else:
    df_entities = df_entities.drop_duplicates(subset=['entity_text', 'entity_type', 'source_url'])
    CSV_PATH = f'{OUTPUT_DIR}/extracted_knowledge.csv'
    df_entities.to_csv(CSV_PATH, index=False, encoding='utf-8')
    print(f'Saved {len(df_entities)} unique entities → {CSV_PATH}')
    print('\nEntity type distribution:')
    print(df_entities['entity_type'].value_counts())

In [ ]:
# ── Export relations CSV ───────────────────────────────────────────────────
df_relations = pd.DataFrame(all_relations)
df_relations = df_relations.drop_duplicates(subset=['subject', 'relation', 'object'])

REL_PATH = f'{OUTPUT_DIR}/extracted_relations.csv'
df_relations.to_csv(REL_PATH, index=False, encoding='utf-8')

print(f'Saved {len(df_relations)} unique relations → {REL_PATH}')
print('\nTop 15 most frequent relation verbs:')
print(df_relations['relation'].value_counts().head(15))

## 📊 6. Summary & Ambiguity Analysis (Report Section)

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────
print('=' * 50)
print('LAB 1 — SUMMARY')
print('=' * 50)
print(f'Pages crawled (Wikipedia):  {len(crawled_pages)}')
print(f'Wikidata records fetched:   {total}')
print(f'Unique entities extracted:  {len(df_entities)}')
print(f'Candidate relations found:  {len(df_relations)}')
print()
print('Entity breakdown:')
for label, count in df_entities['entity_type'].value_counts().items():
    print(f'  {label:<8} {count}')

In [ ]:
# ── Ambiguity Analysis (required for report) ───────────────────────────────
# Find entities where same text appears with different types
ambiguity_cases = (
    df_entities.groupby('entity_text')['entity_type']
    .nunique()
    .reset_index()
    .query('entity_type > 1')
    .sort_values('entity_type', ascending=False)
)

print(f'Entities with ambiguous types: {len(ambiguity_cases)}')
print('\nTop ambiguous entities (same text, different types):')
print(ambiguity_cases.head(10))

# Show examples with their contexts
print('\n--- Ambiguity Case Examples (for report) ---')
for entity in ambiguity_cases.head(3)['entity_text']:
    cases = df_entities[df_entities['entity_text'] == entity][['entity_type', 'sentence', 'source_title']]
    print(f'\nEntity: "{entity}"')
    print(cases.to_string())

## 📈 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Lab 1 — Extraction Overview: FIVB Men\'s Volleyball', fontsize=14, fontweight='bold')

# ── Plot 1: Entity type distribution ──────────────────────────────────────
entity_counts = df_entities['entity_type'].value_counts()
colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0','#00BCD4']
axes[0].bar(entity_counts.index, entity_counts.values, color=colors[:len(entity_counts)])
axes[0].set_title('Entity Type Distribution')
axes[0].set_xlabel('Entity Type')
axes[0].set_ylabel('Count')
for i, v in enumerate(entity_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# ── Plot 2: Entities per source page ──────────────────────────────────────
page_counts = df_entities['source_title'].value_counts().head(10)
short_titles = [t[:25] + '...' if len(t) > 25 else t for t in page_counts.index]
axes[1].barh(short_titles[::-1], page_counts.values[::-1], color='#2196F3')
axes[1].set_title('Top 10 Pages by Entity Count')
axes[1].set_xlabel('Number of Entities')

# ── Plot 3: Relation pattern distribution ─────────────────────────────────
if not df_relations.empty:
    pattern_counts = df_relations['pattern'].value_counts()
    axes[2].pie(pattern_counts.values, labels=pattern_counts.index,
                autopct='%1.1f%%', colors=['#4CAF50','#FF9800','#2196F3'])
    axes[2].set_title('Relation Extraction Patterns')
else:
    axes[2].text(0.5, 0.5, 'No relations\nextracted yet',
                 ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Relation Patterns')

plt.tight_layout()
VIZ_PATH = f'{OUTPUT_DIR}/lab1_overview.png'
plt.savefig(VIZ_PATH, bbox_inches='tight')
plt.show()
print(f'Chart saved → {VIZ_PATH}')


In [ ]:
print('\n✅ Lab 1 complete!')
print('Files saved to Google Drive:')
print(f'  - {OUTPUT_DIR}/crawler_output.jsonl')
print(f'  - {OUTPUT_DIR}/wikidata_output.jsonl')
print(f'  - {OUTPUT_DIR}/extracted_knowledge.csv')
print(f'  - {OUTPUT_DIR}/extracted_relations.csv')
print(f'  - {OUTPUT_DIR}/lab1_overview.png')
print('\nNext step: Lab 2 — KB Construction & Alignment')
